In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install transformers torch scikit-learn accelerate

Cell 2. Now we check if our colab has GPU, If it prints CUDA then we are good, if it pritnts cpu the we have to change it to GPU.

In [3]:
import os
import torch
import numpy as np
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score
)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


3.We label map, Afrisenti uses these three classes we will use.We map them to numbers because this model uses integers not strings.

In [4]:
PROJECT_PATH = '/content/drive/Shareddrives/Cos760'


LBL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LBL = {0: 'negative', 1: 'neutral', 2: 'positive'}


MODEL_NAME = "xlm-roberta-base"

LANGUAGES = ['hausa', 'kinyarwanda']

4. Then we tokenize. Our max max length being 128, to be able to enough words.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['tweet'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

def encode_labels(example):
    example['label'] = LBL2ID[example['label']]
    return example

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

5.cell 5 we define our metrics

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        'f1': f1_score(labels, predictions, average='weighted'),
        'precision': precision_score(labels, predictions, average='weighted', zero_division=0),
        'recall': recall_score(labels, predictions, average='weighted', zero_division=0),
        'accuracy': accuracy_score(labels, predictions)
    }


**6.Fine tune the model on every language.**

In [7]:
import pandas as pd
from torch.utils.data import Dataset

PROJECT_PATH = '/content/drive/Shareddrives/Cos760'

LBL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}

# Custom Dataset class to load from CSV
class SentimentDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tweet = str(self.data.iloc[idx]['cleaned_tweet'])
        label = LBL2ID[self.data.iloc[idx]['label']]

        encoding = self.tokenizer(
            tweet,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

for lang in LANGUAGES:
    print(f"\n{'='*60}")
    print(f"Fine-tuning XLM-R on {lang.upper()}")
    print(f"{'='*60}")

    # Load CSVs
    train_df = pd.read_csv(os.path.join(PROJECT_PATH, 'data/processed', f'{lang}_train_cleaned.csv'))
    val_df   = pd.read_csv(os.path.join(PROJECT_PATH, 'data/processed', f'{lang}_validation_cleaned.csv'))
    test_df  = pd.read_csv(os.path.join(PROJECT_PATH, 'data/processed', f'{lang}_test_cleaned.csv'))

    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

    # Create datasets
    train_dataset = SentimentDataset(train_df, tokenizer)
    val_dataset   = SentimentDataset(val_df, tokenizer)
    test_dataset  = SentimentDataset(test_df, tokenizer)

    # Load model
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=3,
        id2label=ID2LBL,
        label2id=LBL2ID
    ).to(device)

    # Training settings
    training_args = TrainingArguments(
        output_dir=os.path.join(PROJECT_PATH, f'models/xlmr/{lang}'),
        num_train_epochs=5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        warmup_steps=100,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        logging_dir=os.path.join(PROJECT_PATH, f'outputs/metrics/xlmr_{lang}'),
        logging_steps=50,
        fp16=torch.cuda.is_available()
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    print(f"\nTest set results for {lang}:")
    results = trainer.evaluate(test_dataset)
    print(results)

    model.save_pretrained(os.path.join(PROJECT_PATH, f'models/xlmr/{lang}/final'))
    tokenizer.save_pretrained(os.path.join(PROJECT_PATH, f'models/xlmr/{lang}/final'))
    print(f"\n✅ XLM-R fine-tuned and saved for {lang}")


Fine-tuning XLM-R on HAUSA
Train: 14172 | Val: 2677 | Test: 5303


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy
1,0.746399,0.686746,0.693282,0.709466,0.694061,0.694061
2,0.615580,0.655005,0.725321,0.742637,0.729922,0.729922
3,0.562604,0.619298,0.748855,0.750641,0.748226,0.748226
4,0.475197,0.626477,0.757033,0.756950,0.757191,0.757191
5,0.415743,0.664277,0.752524,0.754280,0.751961,0.751961


...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Test set results for kinyarwanda:


{'eval_loss': 0.9380454421043396, 'eval_f1': 0.5661325546848359, 'eval_precision': 0.5838727150571931, 'eval_recall': 0.5750487329434698, 'eval_accuracy': 0.5750487329434698, 'eval_runtime': 2.4194, 'eval_samples_per_second': 424.078, 'eval_steps_per_second': 13.64, 'epoch': 5.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ XLM-R fine-tuned and saved for kinyarwanda


**AfriBERTa fine-tuning code.**

In [ ]:

AFRIBERTA_MODEL = "castorini/afriberta_large"

for lang in LANGUAGES:
    print(f"\n{'='*60}")
    print(f"Fine-tuning AfriBERTa on {lang.upper()}")
    print(f"{'='*60}")

    # Load tokenizer for AfriBERTa
    afriberta_tokenizer = AutoTokenizer.from_pretrained(AFRIBERTA_MODEL)

    # Load CSVs
    train_df = pd.read_csv(os.path.join(PROJECT_PATH, 'data/processed', f'{lang}_train_cleaned.csv'))
    val_df   = pd.read_csv(os.path.join(PROJECT_PATH, 'data/processed', f'{lang}_validation_cleaned.csv'))
    test_df  = pd.read_csv(os.path.join(PROJECT_PATH, 'data/processed', f'{lang}_test_cleaned.csv'))

    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

    # Create datasets using AfriBERTa tokenizer
    train_dataset = SentimentDataset(train_df, afriberta_tokenizer)
    val_dataset   = SentimentDataset(val_df, afriberta_tokenizer)
    test_dataset  = SentimentDataset(test_df, afriberta_tokenizer)

    # Load AfriBERTa with classification head
    model = AutoModelForSequenceClassification.from_pretrained(
        AFRIBERTA_MODEL,
        num_labels=3,
        id2label=ID2LBL,
        label2id=LBL2ID
    ).to(device)

    # Same training settings as XLM-R for fair comparison
    training_args = TrainingArguments(
        output_dir=os.path.join(PROJECT_PATH, f'models/afriberta/{lang}'),
        num_train_epochs=5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        warmup_steps=100,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        logging_dir=os.path.join(PROJECT_PATH, f'outputs/metrics/afriberta_{lang}'),
        logging_steps=50,
        fp16=torch.cuda.is_available()
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    print(f"\nTest set results for {lang}:")
    results = trainer.evaluate(test_dataset)
    print(results)

    model.save_pretrained(os.path.join(PROJECT_PATH, f'models/afriberta/{lang}/final'))
    afriberta_tokenizer.save_pretrained(os.path.join(PROJECT_PATH, f'models/afriberta/{lang}/final'))
    print(f"\n✅ AfriBERTa fine-tuned and saved for {lang}")


Fine-tuning AfriBERTa on HAUSA
Train: 14172 | Val: 2677 | Test: 5303


pytorch_model.bin:   0%|          | 0.00/503M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/503M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: castorini/afriberta_large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Test set results for kinyarwanda:


{'eval_loss': 1.1936004161834717, 'eval_f1': 0.6193261088382385, 'eval_precision': 0.6192234914251821, 'eval_recall': 0.6198830409356725, 'eval_accuracy': 0.6198830409356725, 'eval_runtime': 1.8964, 'eval_samples_per_second': 541.02, 'eval_steps_per_second': 17.401, 'epoch': 5.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ AfriBERTa fine-tuned and saved for kinyarwanda
